# Cinematica — MyCobot 280
### Rol 2: Especialista en Cinematica — Dias
**Problemas que cubre:** P1 (DH), P2 (FK), P3 (IK), P4 (Colisiones)

---
**Orden de ejecucion:**
1. Celda 1 — Conexion al robot
2. Celda 2 — P1: Parametros DH y matrices de transformacion
3. Celda 3 — P2: Cinematica directa (FK)
4. Celda 4 — P2: Verificacion FK vs robot real
5. Celda 5 — P2: Grafica del espacio de trabajo
6. Celda 6 — P3: Cinematica inversa analitica (IK)
7. Celda 7 — P3: Verificacion IK vs robot real
8. Celda 8 — P4: Identificacion de singularidades
9. Celda 9 — Funcion ik_solve para el Rol de Control

In [ ]:
# ============================================================
# CELDA 1 — CONEXION AL ROBOT Y LIBRERIAS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from pymycobot.mycobot import MyCobot
import time

# Conectar al robot
mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)

# Verificar conexion
if mc.is_controller_connected():
    print("Robot conectado correctamente")
else:
    print("ERROR: No se pudo conectar al robot")

print("Angulos actuales:", mc.get_angles())
print("Coordenadas actuales:", mc.get_coords())

In [ ]:
# ============================================================
# CELDA 2 — P1: PARAMETROS DH Y MATRICES DE TRANSFORMACION
#
# Convencion Denavit-Hartenberg (DH):
#   a_i  = longitud del eslabon (mm)
#   d_i  = desplazamiento de la articulacion (mm)
#   alpha_i = angulo de torsion (rad)
#   theta_i = angulo de la articulacion (variable, en rad)
#
# Tabla DH del MyCobot 280 (fuente: datasheet del fabricante)
# Alcance maximo extendido: 280 mm
# ============================================================

# Parametros DH del MyCobot 280
# Columnas: [a_i (mm), d_i (mm), alpha_i (rad)]
# theta_i es la variable articular que se pasa como argumento

DH_PARAMS = [
    # a_i      d_i      alpha_i          Articulacion
    [0,        131.56,  np.pi/2],        # J1 - Base
    [110.4,    0,       0      ],        # J2 - Hombro
    [96.0,     0,       0      ],        # J3 - Codo
    [0,        66.39,  -np.pi/2],        # J4 - Muneca 1
    [0,        73.18,   np.pi/2],        # J5 - Muneca 2
    [0,        48.6,    0      ],        # J6 - Gripper
]

# Rangos articulares del MyCobot 280 (grados)
RANGOS = {
    'J1': (-168, 168),
    'J2': (-135,  90),
    'J3': (-150, 150),
    'J4': (-145, 145),
    'J5': (-165, 165),
    'J6': (-180, 180),
}

def matriz_dh(a, d, alpha, theta):
    """
    Calcula la matriz de transformacion homogenea Ti
    para un eslabon usando la convencion DH.

    Ti = Rz(theta) * Tz(d) * Tx(a) * Rx(alpha)

    Parametros:
        a     : longitud del eslabon (mm)
        d     : desplazamiento articular (mm)
        alpha : angulo de torsion (rad)
        theta : angulo articular (rad)

    Retorna:
        Matriz 4x4 de transformacion homogenea
    """
    ct = np.cos(theta)
    st = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)

    T = np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,   sa,     ca,    d   ],
        [0,   0,      0,     1   ]
    ])
    return T

# Mostrar la tabla DH
print("Tabla de parametros DH — MyCobot 280")
print("=" * 60)
print(f"{'Joint':>6} {'a_i(mm)':>9} {'d_i(mm)':>9} {'alpha_i':>10} {'Rango (deg)':>14}")
print("-" * 60)
nombres = ['J1 Base', 'J2 Hombro', 'J3 Codo', 'J4 Muneca1', 'J5 Muneca2', 'J6 Gripper']
for i, (params, nombre) in enumerate(zip(DH_PARAMS, nombres)):
    a, d, alpha = params
    lo, hi = RANGOS[f'J{i+1}']
    print(f"{nombre:>10} {a:>9.2f} {d:>9.2f} {np.degrees(alpha):>10.1f} {lo:>6} a {hi:>4}")
print("=" * 60)

In [ ]:
# ============================================================
# CELDA 3 — P2: CINEMATICA DIRECTA (FK)
#
# Calcula la posicion del extremo del robot dado un vector
# de angulos articulares [theta1..theta6] en grados.
#
# Formula: T0_6 = T1 * T2 * T3 * T4 * T5 * T6
# La posicion (xe, ye, ze) es la ultima columna de T0_6
# ============================================================

class ForwardKinematics:
    """
    Calcula la cinematica directa del MyCobot 280
    usando la convencion Denavit-Hartenberg.
    """

    def __init__(self, dh_params):
        self.dh = dh_params

    def calcular(self, angulos_deg):
        """
        Calcula la posicion del extremo dado un vector de angulos.

        Parametros:
            angulos_deg: lista de 6 angulos en grados [J1..J6]

        Retorna:
            T0_6: matriz de transformacion homogenea 4x4
            pos:  posicion del extremo [xe, ye, ze] en mm
        """
        # Convertir grados a radianes
        angulos_rad = np.radians(angulos_deg)

        # Iniciar con la identidad
        T0_6 = np.eye(4)

        # Multiplicar las 6 matrices DH: T0_6 = T1 * T2 * ... * T6
        for i in range(6):
            a, d, alpha = self.dh[i]
            theta = angulos_rad[i]
            Ti = matriz_dh(a, d, alpha, theta)
            T0_6 = T0_6 @ Ti  # Multiplicacion matricial

        # La posicion del extremo es la ultima columna (filas 0-2)
        pos = T0_6[:3, 3]  # [xe, ye, ze] en mm
        return T0_6, pos

# Instanciar la clase
fk = ForwardKinematics(DH_PARAMS)

# Verificar con la pose home [0, 0, 0, 0, 0, 0]
T_home, pos_home = fk.calcular([0, 0, 0, 0, 0, 0])
print("Pose home [0, 0, 0, 0, 0, 0]:")
print(f"  Posicion calculada FK: xe={pos_home[0]:.2f}mm, ye={pos_home[1]:.2f}mm, ze={pos_home[2]:.2f}mm")
print("\nMatriz T0_6 en pose home:")
print(np.round(T_home, 3))
print("\nClase ForwardKinematics lista")

In [ ]:
# ============================================================
# CELDA 4 — P2: VERIFICACION FK vs ROBOT REAL
#
# Compara la posicion calculada con FK vs la posicion real
# leida con mc.get_coords() para 5 configuraciones distintas
# Tabula el error en mm
# ============================================================

# 5 configuraciones de prueba (angulos en grados)
CONFIGS_PRUEBA = [
    [0,    0,    0,    0,    0,   -45],   # Config 1: pose inicial
    [45,   0,    0,    0,    0,   -45],   # Config 2: base girada
    [0,  -45,  -45,   0,    0,   -45],   # Config 3: brazo inclinado
    [-30, -60,   0,    0,    0,   -45],   # Config 4: posicion lateral
    [30,  -30, -60,   10,   20,  -45],   # Config 5: posicion mixta
]

print("Tabla FK: Posicion calculada vs posicion real")
print("=" * 75)
print(f"{'Config':>7} {'FK xe':>8} {'FK ye':>8} {'FK ze':>8} {'Real x':>8} {'Real y':>8} {'Real z':>8} {'Error':>7}")
print("-" * 75)

for i, config in enumerate(CONFIGS_PRUEBA):
    # Calcular FK
    _, pos_calc = fk.calcular(config)
    xe, ye, ze = pos_calc

    # Mover el robot a la configuracion
    mc.send_angles(config, 20)
    time.sleep(3)

    # Leer posicion real del robot
    coords_real = mc.get_coords()

    if coords_real:
        xr, yr, zr = coords_real[0], coords_real[1], coords_real[2]
        # Error euclidiano en mm
        error = np.sqrt((xe-xr)**2 + (ye-yr)**2 + (ze-zr)**2)
        print(f"  C{i+1}    {xe:>8.1f} {ye:>8.1f} {ze:>8.1f} {xr:>8.1f} {yr:>8.1f} {zr:>8.1f} {error:>7.2f}mm")
    else:
        print(f"  C{i+1}    {xe:>8.1f} {ye:>8.1f} {ze:>8.1f}   Error al leer robot")

print("=" * 75)

# Volver a pose inicial al terminar
mc.send_angles([0, 0, 0, 0, 0, -45], 20)
time.sleep(3)

In [ ]:
# ============================================================
# CELDA 5 — P2: GRAFICA DEL ESPACIO DE TRABAJO
#
# Genera una nube de puntos 2D en el plano XZ calculando
# la FK para miles de configuraciones aleatorias dentro
# de los rangos articulares del robot
# ============================================================

print("Calculando espacio de trabajo (puede tardar unos segundos)...")

# Numero de muestras aleatorias
N_MUESTRAS = 5000

puntos_x = []
puntos_z = []

for _ in range(N_MUESTRAS):
    # Generar configuracion aleatoria dentro de los rangos
    config_aleatoria = [
        np.random.uniform(*RANGOS['J1']),
        np.random.uniform(*RANGOS['J2']),
        np.random.uniform(*RANGOS['J3']),
        np.random.uniform(*RANGOS['J4']),
        np.random.uniform(*RANGOS['J5']),
        np.random.uniform(*RANGOS['J6']),
    ]

    # Calcular posicion del extremo con FK
    _, pos = fk.calcular(config_aleatoria)
    puntos_x.append(pos[0])
    puntos_z.append(pos[2])

# Graficar nube de puntos en plano XZ
plt.figure(figsize=(8, 8))
plt.scatter(puntos_x, puntos_z, s=0.5, alpha=0.3, color='steelblue')
plt.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
plt.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('X (mm)')
plt.ylabel('Z (mm)')
plt.title('Espacio de trabajo alcanzable — MyCobot 280 (plano XZ)')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.savefig('/tmp/espacio_trabajo_XZ.png', dpi=150)
plt.show()

print(f"Grafica guardada en /tmp/espacio_trabajo_XZ.png")
print(f"Puntos graficados: {N_MUESTRAS}")

In [ ]:
# ============================================================
# CELDA 6 — P3: CINEMATICA INVERSA ANALITICA (IK)
#
# Implementa la IK analitica simplificada para los 3
# primeros joints usando modelo planar 2R + base:
#
# theta1 = atan2(y, x)              -- base apunta al objetivo
# theta3 = atan2(+-sqrt(1-cos2), cos3)  -- angulo del codo
# theta2 = atan2(z', r) - atan2(L3*sin3, L2+L3*cos3)
#
# Parametros del modelo planar:
#   L2 = 110.4 mm  (longitud eslabon 2)
#   L3 = 96.0  mm  (longitud eslabon 3)
#   d1 = 131.56 mm (altura de la base)
# ============================================================

class InverseKinematics:
    """
    Calcula la cinematica inversa analitica simplificada
    para los 3 primeros joints del MyCobot 280.
    Los joints 4, 5, 6 se fijan a cero (orientacion neutra).
    """

    def __init__(self):
        self.L2 = 110.4   # Longitud eslabon 2 (mm)
        self.L3 = 96.0    # Longitud eslabon 3 (mm)
        self.d1 = 131.56  # Altura de la base (mm)

    def calcular(self, x, y, z):
        """
        Calcula los angulos J1, J2, J3 para alcanzar (x, y, z).

        Parametros:
            x, y, z: posicion objetivo en mm

        Retorna:
            [theta1, theta2, theta3, 0, 0, 0] en grados
            None si la posicion no es alcanzable
        """
        # J1: angulo de la base — apunta hacia el objetivo en XY
        theta1 = np.degrees(np.arctan2(y, x))

        # Distancia radial en el plano XY
        r  = np.sqrt(x**2 + y**2)

        # Altura relativa a la base
        z_prima = z - self.d1

        # Verificar si la posicion es alcanzable
        distancia = np.sqrt(r**2 + z_prima**2)
        if distancia > (self.L2 + self.L3):
            print(f"Posicion ({x}, {y}, {z}) fuera del espacio de trabajo")
            print(f"  Distancia requerida: {distancia:.1f}mm")
            print(f"  Alcance maximo: {self.L2 + self.L3:.1f}mm")
            return None

        # J3: angulo del codo (ley de cosenos)
        cos_theta3 = (r**2 + z_prima**2 - self.L2**2 - self.L3**2) / (2 * self.L2 * self.L3)
        cos_theta3 = np.clip(cos_theta3, -1, 1)  # Evitar errores numericos

        # Solucion codo arriba (signo positivo)
        sin_theta3 = np.sqrt(1 - cos_theta3**2)
        theta3 = np.degrees(np.arctan2(sin_theta3, cos_theta3))

        # J2: angulo del hombro
        theta2 = np.degrees(
            np.arctan2(z_prima, r) -
            np.arctan2(self.L3 * sin_theta3, self.L2 + self.L3 * cos_theta3)
        )

        # J4, J5, J6 fijos a 0 (orientacion neutra)
        return [round(theta1, 2), round(theta2, 2), round(theta3, 2), 0, 0, 0]

# Instanciar la clase
ik = InverseKinematics()

# Prueba rapida
angulos = ik.calcular(150, 0, 200)
print("Prueba IK para posicion (150, 0, 200):")
print(f"  Angulos calculados: {angulos}")
print("\nClase InverseKinematics lista")

In [ ]:
# ============================================================
# CELDA 7 — P3: VERIFICACION IK vs ROBOT REAL
#
# Compara la IK analitica vs la solucion del API del robot
# para 3 posiciones cartesianas distintas.
# Tabula el error en grados entre ambas soluciones.
# ============================================================

# 3 posiciones cartesianas de prueba [x, y, z, rx, ry, rz]
# rx, ry, rz = orientacion del extremo (fija para comparacion justa)
POSICIONES_IK = [
    [150,   0,  200, -90, -45, -90],   # Posicion 1: frente al robot
    [200, -50,  180, -90, -45, -90],   # Posicion 2: lateral derecha
    [100,  50,  250, -90, -45, -90],   # Posicion 3: lateral izquierda alta
]

print("Tabla IK: Analitica calculada vs solucion del API")
print("=" * 70)
print(f"{'Pos':>4} {'J1 calc':>9} {'J1 API':>9} {'J2 calc':>9} {'J2 API':>9} {'J3 calc':>9} {'J3 API':>9}")
print("-" * 70)

for i, pos in enumerate(POSICIONES_IK):
    x, y, z = pos[0], pos[1], pos[2]

    # IK analitica (nuestra implementacion)
    angulos_calc = ik.calcular(x, y, z)

    # IK del API (solucion del robot)
    mc.send_coords(pos, 20, 1)
    time.sleep(4)
    angulos_api = mc.get_angles()

    if angulos_calc and angulos_api:
        j1c, j2c, j3c = angulos_calc[0], angulos_calc[1], angulos_calc[2]
        j1a, j2a, j3a = angulos_api[0],  angulos_api[1],  angulos_api[2]
        print(f"  P{i+1}  {j1c:>9.2f} {j1a:>9.2f} {j2c:>9.2f} {j2a:>9.2f} {j3c:>9.2f} {j3a:>9.2f}")

        # Calcular error en grados
        err1 = abs(j1c - j1a)
        err2 = abs(j2c - j2a)
        err3 = abs(j3c - j3a)
        print(f"       Error J1={err1:.2f} deg | Error J2={err2:.2f} deg | Error J3={err3:.2f} deg")
    else:
        print(f"  P{i+1}  Error al leer datos")

print("=" * 70)

# Volver a pose inicial
mc.send_angles([0, 0, 0, 0, 0, -45], 20)
time.sleep(3)

In [ ]:
# ============================================================
# CELDA 8 — P4: SINGULARIDADES Y COLISIONES
#
# Identifica configuraciones singulares donde la IK
# no tiene solucion unica o el robot pierde un grado
# de libertad, y posiciones fuera del espacio de trabajo.
#
# Singularidades del MyCobot 280:
#   1. Singularidad de muneca: J5 = 0 (ejes J4 y J6 alineados)
#   2. Singularidad de codo:   brazo completamente extendido
#   3. Singularidad de hombro: extremo sobre eje J1
# ============================================================

class CollisionChecker:
    """
    Verifica configuraciones singulares y posiciones
    fuera del espacio de trabajo del MyCobot 280.
    """

    def __init__(self, rangos, z_min=125):
        self.rangos = rangos
        self.z_min  = z_min         # Altura minima sobre la mesa (mm)
        self.L2     = 110.4         # Longitud eslabon 2
        self.L3     = 96.0          # Longitud eslabon 3
        self.alcance_max = self.L2 + self.L3  # 206.4 mm

    def es_singular(self, angulos_deg):
        """
        Verifica si la configuracion es singular.
        Retorna (True, descripcion) o (False, 'OK')
        """
        j5 = angulos_deg[4]

        # Singularidad de muneca: J5 cercano a 0
        if abs(j5) < 2.0:
            return True, "Singularidad de muneca (J5 ~ 0): ejes J4 y J6 alineados"

        # Singularidad de codo: brazo extendido (theta3 ~ 0 o 180)
        j3 = angulos_deg[2]
        if abs(j3) < 2.0 or abs(abs(j3) - 180) < 2.0:
            return True, "Singularidad de codo (J3 ~ 0 o 180): brazo extendido"

        return False, "OK"

    def es_alcanzable(self, x, y, z):
        """
        Verifica si la posicion (x, y, z) esta dentro
        del espacio de trabajo alcanzable.
        """
        r = np.sqrt(x**2 + y**2)
        z_prima = z - 131.56  # Restar altura de la base
        distancia = np.sqrt(r**2 + z_prima**2)

        if distancia > self.alcance_max:
            return False, f"Fuera de alcance: {distancia:.1f}mm > {self.alcance_max:.1f}mm"
        if z < self.z_min:
            return False, f"Demasiado cerca de la mesa: z={z}mm < {self.z_min}mm"

        return True, "Alcanzable"

    def verificar_angulos(self, angulos_deg):
        """
        Verifica que los angulos esten dentro de los
        limites fisicos del robot.
        """
        for i, ang in enumerate(angulos_deg):
            lo, hi = self.rangos[f'J{i+1}']
            if not (lo <= ang <= hi):
                return False, f"J{i+1}={ang} fuera de rango [{lo}, {hi}]"
        return True, "OK"

# Instanciar la clase
checker = CollisionChecker(RANGOS)

# Pruebas de singularidades
print("Pruebas de singularidades y alcanzabilidad")
print("=" * 55)

casos = [
    ([0, -30, 0, 0, 0, 0],    "J5=0 singular"),
    ([0, 0, 0, 0, 20, 0],     "Configuracion normal"),
    ([0, -90, -90, 0, 20, 0], "Brazo inclinado"),
]

for angulos, desc in casos:
    singular, msg = checker.es_singular(angulos)
    valido, msg2  = checker.verificar_angulos(angulos)
    print(f"\n{desc}: {angulos}")
    print(f"  Singular: {singular} — {msg}")
    print(f"  Angulos validos: {valido} — {msg2}")

print("\n" + "=" * 55)
print("Prueba de posiciones alcanzables")
print("=" * 55)

posiciones = [
    (150, 0, 200, "Posicion normal"),
    (400, 0, 200, "Fuera de alcance"),
    (100, 0,  50, "Demasiado baja"),
]

for x, y, z, desc in posiciones:
    alcanzable, msg = checker.es_alcanzable(x, y, z)
    print(f"  ({x}, {y}, {z}) — {desc}: {msg}")

print("\nClase CollisionChecker lista")

In [ ]:
# ============================================================
# CELDA 9 — FUNCION ik_solve PARA EL ROL DE CONTROL
#
# Funcion requerida por Alex (Rol de Control) para calcular
# los angulos de agarre dado un punto (x, y, z) detectado
# por la vision de Aaron.
#
# Entrega: ik_solve(x, y, z) -> [J1, J2, J3, J4, J5, J6]
# ============================================================

def ik_solve(x, y, z):
    """
    Calcula los angulos articulares para llevar el extremo
    del MyCobot 280 a la posicion (x, y, z).

    Primero verifica que la posicion sea alcanzable,
    luego calcula la IK analitica simplificada.

    Parametros:
        x, y, z: posicion objetivo en mm

    Retorna:
        [J1, J2, J3, J4, J5, J6] en grados
        None si la posicion no es alcanzable
    """
    # Verificar que la posicion sea alcanzable
    alcanzable, msg = checker.es_alcanzable(x, y, z)
    if not alcanzable:
        print(f"ik_solve: posicion no alcanzable — {msg}")
        return None

    # Calcular IK analitica
    angulos = ik.calcular(x, y, z)
    if angulos is None:
        return None

    # Verificar que los angulos esten dentro de los limites
    valido, msg = checker.verificar_angulos(angulos)
    if not valido:
        print(f"ik_solve: angulos fuera de limite — {msg}")
        return None

    return angulos

# Prueba de la funcion ik_solve
print("Prueba de ik_solve")
print("=" * 45)

casos = [
    (150,   0, 200),   # Posicion alcanzable
    (400,   0, 200),   # Fuera de alcance
    (100, -50, 150),   # Posicion lateral
]

for x, y, z in casos:
    resultado = ik_solve(x, y, z)
    print(f"\nik_solve({x}, {y}, {z}):")
    if resultado:
        print(f"  Angulos: {resultado}")
    else:
        print("  No alcanzable")

print("\n" + "=" * 45)
print("Funciones disponibles:")
print("  fk.calcular(angulos_deg)  -> T0_6, posicion")
print("  ik.calcular(x, y, z)     -> angulos")
print("  ik_solve(x, y, z)        -> angulos validados")
print("  checker.es_singular()    -> True/False")
print("  checker.es_alcanzable()  -> True/False")